# GISLR — Semantic Sign-Pair Similarity (Angles + Kinematics)

**What this notebook does.** A **dataset-stage, no-training** analysis
targeting a specific, already-documented question:
[`docs/reports/plateau-diagnosis.md`](../../docs/reports/plateau-diagnosis.md)
§6 found that a binary probe on **pooled position/velocity** features
(mean/std/min/max over frames, ME-126 xy) sits at only 0.463 accuracy for
`awake`/`wake` (chance) and a median of 0.628 across 7 semantically-confused
pairs — and explicitly flagged that this "bounds time-pooled summaries, not
the landmarks": pooling destroys the ordering/trajectory shape that
distinguishes e.g. `awake` (repeated motion) from `wake` (single motion).

This notebook re-runs that same binary-separability-probe methodology (5-fold
CV, same as §6) on the **16 documented confusable pairs**
(`plateau-diagnosis.md` §1, replicated across 31 canonical trained-model
runs and two independent architectures/feature pipelines), but using the
richer `landmark_kinematics_v1` feature set from the feature-discriminability
track (TODO §3.4) — 28 joint angles, frame-gap-aware position/speed, jitter,
rolling variance, relational distances — instead of the old raw xy
position/velocity summaries. **Question: do joint angles (which §6 never
had) rescue any of these pairs, or does mean/std pooling remain the
bottleneck regardless of what's pooled?**

**Pipeline stage vs. diagnostic.** No training. Reuses
`sb.recognize.interp.kinematics`/`discriminability` unchanged (TODO §3.4) —
this notebook only picks a different, hand-picked class list (the known
confusable pairs, plus a random-pair control, mirroring
`plateau-diagnosis.md` §2's random-pair-control convention) instead of a
random class sample.

**Artifacts**

| Path | Content |
|---|---|
| `data/cache/gislr/pair_similarity/pairs/*.parquet` | per-video descriptors for the 16 documented pairs + control pairs |
| `data/cache/gislr/pair_similarity/pairs_manifest.json` | resumable manifest |
| `docs/reports/pair-similarity.md` | write-up, **after** this notebook runs |

**Design decisions**

- Pairs and their symmetric confusion rates are taken verbatim from
  `plateau-diagnosis.md` §1 (31-run aggregate) — not re-derived here.
- `finger`/`wait` and `animal`/`have` are in the list despite **not** being
  semantically related (per that report, confused as heavily as the
  near-synonyms without meaning in common) — kept for comparability with §6,
  which also included them.
- **Random-pair control**: `N_CONTROL_PAIRS` pairs of classes sampled at
  random (seed-fixed) from the remaining GISLR vocabulary, scored the exact
  same way — the semantic pairs' probe accuracy only means something next to
  what an arbitrary pair looks like.
- Per pair: 5-fold CV probe accuracy (`disc.probe_classifier_cv`, matching
  §6's methodology) on the full feature set, **and** broken down by feature
  type (angle-only / position-only / speed-only / etc.) to see which feature
  type, if any, moves a given pair off its §6 baseline.


In [ ]:
# ============================================================
# Imports & config
# ============================================================
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, gislr_dir
from sb.recognize.interp import discriminability as disc
from sb.recognize.interp import kinematics as kin

DATA_DIR = gislr_dir()
PS_CACHE_DIR = CACHE_DIR / "gislr" / "pair_similarity"
PS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
SAVGOL_WINDOW = 7
SAVGOL_POLYORDER = 2
ROLLING_VAR_WINDOW = 5
CHUNK_SIZE = 200        # videos per resumable unit
CV_FOLDS = 5            # matches plateau-diagnosis.md §6's 5-fold CV

# The 16 documented confusable pairs + their 31-run aggregate symmetric
# confusion rate (plateau-diagnosis.md §1). `sleep` appears in two pairs.
SEMANTIC_PAIRS = [
    ("awake", "wake", 0.839),
    ("lips", "mouth", 0.533),
    ("hear", "listen", 0.411),
    ("pen", "pencil", 0.404),
    ("gift", "give", 0.367),
    ("cut", "scissors", 0.357),
    ("finger", "wait", 0.343),   # not semantically related -- kept for comparability with §6
    ("stay", "that", 0.336),
    ("duck", "goose", 0.330),
    ("animal", "have", 0.283),   # not semantically related
    ("cat", "kitty", 0.271),
    ("sleep", "sleepy", 0.263),
    ("nap", "sleep", 0.260),
    ("bed", "bedroom", 0.212),
    ("dry", "dryer", 0.209),
    ("look", "see", 0.194),
]

# plateau-diagnosis.md §6's own position/velocity/both probe numbers (single
# split, ME-126 xy time-pooled mean/std/min/max), for direct comparison --
# only 7 of the 16 pairs were probed there.
SEC6_PROBE = {
    ("awake", "wake"): {"position": 0.463, "velocity": 0.506, "both": 0.463},
    ("lips", "mouth"): {"position": 0.674, "velocity": 0.588, "both": 0.638},
    ("pen", "pencil"): {"position": 0.632, "velocity": 0.556, "both": 0.597},
    ("cut", "scissors"): {"position": 0.687, "velocity": 0.638, "both": 0.698},
    ("hear", "listen"): {"position": 0.754, "velocity": 0.612, "both": 0.728},
    ("gift", "give"): {"position": 0.754, "velocity": 0.693, "both": 0.736},
    ("finger", "wait"): {"position": 0.790, "velocity": 0.692, "both": 0.768},
}

N_CONTROL_PAIRS = 8

print("DATA_DIR:", DATA_DIR)
print("cache   :", PS_CACHE_DIR.resolve())
print(f"{len(SEMANTIC_PAIRS)} semantic pairs, {N_CONTROL_PAIRS} random control pairs")


## 1. Reusable core — class list, descriptor computation, resumable driver

In [ ]:
# ============================================================
# train.csv + pair/control class list
# ============================================================
train_csv = pd.read_csv(DATA_DIR / "train.csv")
print(f"{len(train_csv):,} videos - {train_csv['sign'].nunique()} signs")

semantic_classes = sorted({c for a, b, _ in SEMANTIC_PAIRS for c in (a, b)})

rng = np.random.default_rng(SEED)
remaining_signs = sorted(set(train_csv["sign"].unique()) - set(semantic_classes))
control_pool = rng.choice(remaining_signs, size=2 * N_CONTROL_PAIRS, replace=False)
control_pairs = [(control_pool[2 * i], control_pool[2 * i + 1], None)
                 for i in range(N_CONTROL_PAIRS)]
print("Control pairs:", [(a, b) for a, b, _ in control_pairs])

all_pairs = SEMANTIC_PAIRS + control_pairs
all_classes = sorted({c for a, b, _ in all_pairs for c in (a, b)})
pair_df = (train_csv[train_csv["sign"].isin(all_classes)]
           .sort_values("uid").reset_index(drop=True))
print(f"{len(all_classes)} classes total, {len(pair_df):,} videos")


def npz_path(row: pd.Series) -> Path:
    return DATA_DIR / row["npz_relpath"]


def descriptors_for_rows(rows: pd.DataFrame) -> pd.DataFrame:
    """One batch of train.csv rows -> tidy per-video descriptor DataFrame."""
    out = []
    for _, row in rows.iterrows():
        desc = kin.load_video_descriptors(
            npz_path(row),
            savgol_window=SAVGOL_WINDOW,
            savgol_polyorder=SAVGOL_POLYORDER,
            rolling_var_window=ROLLING_VAR_WINDOW,
        )
        desc["uid"] = row["uid"]
        desc["sign"] = row["sign"]
        out.append(desc)
    return pd.DataFrame(out)


def feature_type(name: str) -> str:
    if name.startswith("angle_"):
        return "angle"
    if name.startswith("relational_"):
        return "relational"
    if name.startswith("jitter_rms_"):
        return "jitter"
    if name.startswith("rolling_var_speed_"):
        return "rolling_var"
    if "_detection_rate" in name:
        return "detection_rate"
    if "_speed_" in name:
        return "speed"
    if name in ("n_frames", "valid_frame_frac"):
        return "meta"
    return "position"


In [ ]:
# ============================================================
# Manifest helpers + resumable driver (same pattern as feature-discriminability.ipynb)
# ============================================================
def _manifest_path(scope: str) -> Path:
    return PS_CACHE_DIR / f"{scope}_manifest.json"


def _unit_path(scope: str, unit_id: str) -> Path:
    return PS_CACHE_DIR / scope / f"{unit_id}.parquet"


def load_manifest(scope: str) -> dict:
    p = _manifest_path(scope)
    return json.loads(p.read_text()) if p.exists() else {}


def save_manifest(scope: str, manifest: dict) -> None:
    p = _manifest_path(scope)
    tmp = p.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(manifest, indent=2))
    os.replace(tmp, p)


def process_units(scope: str, unit_ids: list, process_fn, desc: str | None = None) -> dict:
    """Artifact written FIRST, unit marked `done` after; `done` skipped, `failed` retried."""
    (PS_CACHE_DIR / scope).mkdir(parents=True, exist_ok=True)
    manifest = load_manifest(scope)
    n_skip = n_ok = n_fail = 0
    for uid in tqdm([str(u) for u in unit_ids], desc=desc or scope):
        entry = manifest.get(uid)
        if entry and entry["status"] == "done" and Path(entry["artifact"]).exists():
            n_skip += 1
            continue
        try:
            out = process_fn(uid)
            artifact = _unit_path(scope, uid)
            out.to_parquet(artifact, index=False)
            manifest[uid] = {"status": "done",
                             "timestamp": datetime.now(timezone.utc).isoformat(),
                             "artifact": str(artifact)}
            n_ok += 1
        except Exception as e:
            manifest[uid] = {"status": "failed",
                             "timestamp": datetime.now(timezone.utc).isoformat(),
                             "artifact": None, "error": repr(e)}
            n_fail += 1
        save_manifest(scope, manifest)
    print(f"[{scope}] done={n_ok}  skipped={n_skip}  failed={n_fail}")
    if n_fail:
        print("failed units:", {k: v["error"] for k, v in manifest.items() if v["status"] == "failed"})
    return manifest


def load_cached_units(scope: str, unit_ids: list) -> pd.DataFrame:
    frames = [pd.read_parquet(_unit_path(scope, str(u)))
              for u in unit_ids if _unit_path(scope, str(u)).exists()]
    return pd.concat(frames, ignore_index=True)


## 2. Compute descriptors for every pair/control class (one resumable pass)

In [ ]:
# ============================================================
# Chunk all 32+2*N_CONTROL_PAIRS classes' videos and compute descriptors once
# ============================================================
chunks = [pair_df.iloc[i:i + CHUNK_SIZE] for i in range(0, len(pair_df), CHUNK_SIZE)]
chunk_ids = [str(i) for i in range(len(chunks))]


def _pairs_unit(chunk_id: str) -> pd.DataFrame:
    return descriptors_for_rows(chunks[int(chunk_id)])


process_units("pairs", chunk_ids, _pairs_unit, desc="pair classes")
pairs_desc = load_cached_units("pairs", chunk_ids)
feature_cols = [c for c in pairs_desc.columns if c not in ("uid", "sign")]
print(pairs_desc.shape)
print(pairs_desc["sign"].value_counts().sort_index())


## 3. Per-pair binary separability probe (5-fold CV)

Directly comparable to `plateau-diagnosis.md` §6: same CV scheme, same
"how well can a linear probe on pooled per-video features tell these two
signs apart" question — different feature set (angles + kinematics instead
of raw xy position/velocity).

In [ ]:
# ============================================================
# Per pair: full-feature probe, plus a per-type breakdown
# ============================================================
def pair_matrix(a: str, b: str):
    sub = pairs_desc[pairs_desc["sign"].isin([a, b])]
    X, names = disc.descriptor_matrix(sub[feature_cols].to_dict("records"))
    y = sub["sign"].to_numpy()
    return X, y, names


types = np.array([feature_type(c) for c in sorted(feature_cols)])
rows = []
for a, b, conf_rate in all_pairs:
    X, y, names = pair_matrix(a, b)
    cv = disc.probe_classifier_cv(X, y, seed=SEED, n_splits=CV_FOLDS)
    row = {
        "pair": f"{a}/{b}", "a": a, "b": b,
        "confusion_rate": conf_rate, "is_control": conf_rate is None,
        "n_videos": len(y),
        "acc_all": cv["acc_mean"], "acc_all_std": cv["acc_std"],
    }
    for t in ("angle", "position", "speed", "relational", "jitter", "rolling_var"):
        mask = types == t
        if mask.sum() == 0:
            continue
        cv_t = disc.probe_classifier_cv(X[:, mask], y, seed=SEED, n_splits=CV_FOLDS)
        row[f"acc_{t}"] = cv_t["acc_mean"]
    rows.append(row)

pair_results = pd.DataFrame(rows)
print(pair_results.sort_values("confusion_rate", ascending=False, na_position="last")
      .to_string(index=False))


## 4. Does confusion rate track probe accuracy? And vs. §6's numbers

In [ ]:
# ============================================================
# corr(confusion_rate, probe accuracy) -- the same check plateau-diagnosis.md
# §6 ran (-0.72 there, on 7 pairs / raw xy position+velocity)
# ============================================================
semantic_results = pair_results[~pair_results["is_control"]]
rho = semantic_results["confusion_rate"].corr(semantic_results["acc_all"], method="spearman")
pear = semantic_results["confusion_rate"].corr(semantic_results["acc_all"])
print(f"Spearman rho(confusion_rate, acc_all) = {rho:.3f}  (16 semantic pairs)")
print(f"Pearson  r  (confusion_rate, acc_all) = {pear:.3f}")
print()
print(f"Control pairs (no semantic relationship, random): "
      f"acc_all mean={pair_results[pair_results['is_control']]['acc_all'].mean():.3f}, "
      f"min={pair_results[pair_results['is_control']]['acc_all'].min():.3f}")
print(f"Semantic pairs: acc_all median={semantic_results['acc_all'].median():.3f}, "
      f"min={semantic_results['acc_all'].min():.3f} ({semantic_results.loc[semantic_results['acc_all'].idxmin(), 'pair']})")

# side-by-side against §6's single-split position/velocity/both numbers
compare_rows = []
for a, b, conf in SEMANTIC_PAIRS:
    key = (a, b)
    if key not in SEC6_PROBE:
        continue
    ours = pair_results[(pair_results["a"] == a) & (pair_results["b"] == b)].iloc[0]
    compare_rows.append({
        "pair": f"{a}/{b}", "confusion_rate": conf,
        "sec6_position": SEC6_PROBE[key]["position"],
        "sec6_velocity": SEC6_PROBE[key]["velocity"],
        "sec6_both": SEC6_PROBE[key]["both"],
        "ours_all_features": ours["acc_all"],
        "ours_angle_only": ours.get("acc_angle", float("nan")),
    })
compare_df = pd.DataFrame(compare_rows)
print()
print(compare_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(semantic_results["confusion_rate"], semantic_results["acc_all"], label="semantic pairs")
for _, r in semantic_results.iterrows():
    ax.annotate(r["pair"], (r["confusion_rate"], r["acc_all"]), fontsize=7,
                xytext=(3, 3), textcoords="offset points")
ax.axhline(pair_results[pair_results["is_control"]]["acc_all"].mean(),
           color="gray", linestyle="--", label="control-pair mean acc")
ax.set_xlabel("31-run aggregate symmetric confusion rate (plateau-diagnosis.md §1)")
ax.set_ylabel("5-fold CV probe accuracy, all features (this notebook)")
ax.set_title("Confusion rate vs. new-feature-set separability")
ax.legend()
plt.tight_layout()
plt.show()


## 5. Which feature type moves a pair off its baseline?

Per semantic pair, `acc_angle` / `acc_position` / `acc_speed` / ... vs
`acc_all` — if a pair's `acc_angle` clears its `acc_all` or its §6 position
baseline by a wide margin, joint angles are carrying that pair's
separability; if every feature type sits near the same (low) number, the
pair is hard regardless of what is pooled — consistent with §6's own
reading that pooling itself, not the choice of channel, is the ceiling.

In [ ]:
# ============================================================
# Per-pair feature-type comparison table + top-F distinguishing features
# ============================================================
type_cols = [c for c in pair_results.columns
             if c.startswith("acc_") and c not in ("acc_all", "acc_all_std")]
print(semantic_results[["pair", "confusion_rate"] + type_cols]
      .sort_values("confusion_rate", ascending=False).to_string(index=False))

print()
print("Top 8 distinguishing features (F-ratio) for the 3 highest-confusion pairs:")
for a, b, conf in sorted(SEMANTIC_PAIRS, key=lambda p: -p[2])[:3]:
    X, y, names = pair_matrix(a, b)
    f_tbl = disc.f_ratio(X, y, names)
    print(f"\n{a}/{b} (confusion={conf}):")
    print(f_tbl.head(8).to_string(index=False))


## 6. Verdict

Fill in after running: does `corr(confusion_rate, acc_all)` come out negative
(as §6 found, -0.72) with this richer feature set too, and is the magnitude
similar or weaker/stronger? Does `awake`/`wake` (confusion 0.839, §6's
position-probe floor 0.463) move at all under the full feature set, or under
angles specifically? Do any pairs' `acc_angle` clear their `acc_all`/§6
baseline meaningfully, suggesting handshape (captured by angles) — as
opposed to trajectory/position — is where that specific pair's real
distinction lives?

In [ ]:
# ============================================================
# Verdict tables (fill in narrative above after running)
# ============================================================
print("Pairs where acc_all beats the control-pair mean by the widest margin (easiest):")
control_mean = pair_results[pair_results["is_control"]]["acc_all"].mean()
sr = semantic_results.copy()
sr["margin_over_control"] = sr["acc_all"] - control_mean
print(sr.sort_values("margin_over_control", ascending=False)
      [["pair", "confusion_rate", "acc_all", "margin_over_control"]].to_string(index=False))
print()
print("Pairs closest to chance (hardest to separate on ANY pooled feature):")
print(sr.sort_values("acc_all")[["pair", "confusion_rate", "acc_all"] + type_cols]
      .head(5).to_string(index=False))


## 7. Follow-ups & what this notebook deliberately does NOT do

- **Still time-pooled** (mean+std over frames) — per `plateau-diagnosis.md`
  §6's own caveat, pooling destroys ordering/trajectory shape (exactly where
  `awake`'s repeated motion differs from `wake`'s single motion). A per-frame
  sequence-model probe (not a pooled linear one) would be needed to test
  whether angles help *there*; this notebook only tests whether a richer
  *pooled* channel set helps a *linear* probe.
- **`worst_pair_overlap` from `sb.recognize.interp.discriminability` is
  identical to `mean_overlap` for a 2-class pair** (only one pair exists) —
  not separately reported here, `acc_all`/F-ratio carry the signal instead.
- **No trained-model confirmation** — a pair whose `acc_angle` clears its
  baseline here is a candidate for adding explicit angle-channel inputs to a
  trained GRU/LSTM run, not a proven fix (mirrors TODO §3.0→§3.1's pattern).
- **31-class-total scope** (16 pairs + 8 control) — not the global 250-class
  scope; this notebook is specifically about the already-identified
  confusable pairs, not a general landmark/feature ranking (that's TODO
  §3.4's job).
